# Reflection with LangChain (Tweet Generator)

In [1]:
# Install all required packages
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# pip install -q langsmith

## Generate

In [2]:
# loading the API keys
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [3]:
# importing the necessary libraries
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

In [ ]:
# creating a chat prompt template
generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            '''You are a Twitter expert assigned to craft outstanding tweets.
            Generate the most engaging and impactful tweet possible based on the user's request.
            If the user provides feedback, refine and enhance your previous attempts accordingly for maximum engagement.''',
        ),
        MessagesPlaceholder(variable_name='messages'),
    ]
)

llm = ChatOpenAI(model_name='gpt-5.1', temperature=0.7)

# using LCEL to create the generate_chain
generate_chain  = generation_prompt | llm

In [5]:
generate_chain

ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.mes

In [6]:
tweet = ''
request = HumanMessage(
    content='FIFA World Cup 26'
)

for chunk in generate_chain.stream(
    {'messages': [request]}
): 
    print(chunk.content, end='')
    tweet += chunk.content

🌍⚽️ Get ready, football fans! The FIFA World Cup 2026 is just around the corner, and it's set to be the biggest and most exciting tournament yet! 🇺🇸🇨🇦🇲🇽 Who are you cheering for? Let the countdown to glory begin! 🎉🏆 #FIFAWorldCup2026 #WorldCup #FootballFever

## Reflect and Repeat

In [7]:
reflection_prompt = ChatPromptTemplate.from_messages(
    [
        (
            'system',
            '''You are a Twitter influencer known for your engaging content and sharp insights.
            Review and critique the user’s tweet.
            Provide constructive feedback, focusing on enhancing its depth, style, and overall impact.
            Offer specific suggestions to make the tweet more compelling and engaging for their audience.'''
        ),
        MessagesPlaceholder(variable_name='messages'),
    ]
)

reflect_chain  = reflection_prompt | llm

In [8]:
reflect_chain

ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchain_core.mes

In [9]:
reflection = ''
# streaming the response
for chunk in reflect_chain.stream(
    {'messages': [request, HumanMessage(content=tweet)]}
):
    print(chunk.content, end='')
    reflection += chunk.content


This tweet has great energy and enthusiasm! The use of emojis adds a fun touch, and you've effectively set the stage for excitement around the FIFA World Cup 2026. Here are some suggestions to enhance its depth, style, and overall impact:

1. **Add a Personal Touch:** Share your own thoughts or predictions about the tournament. This invites engagement and can spark conversation. For example, you might say, "I'm rooting for [your favorite team]! What about you?"

2. **Include a Question:** Asking a more specific question can drive engagement. Instead of just "Who are you cheering for?" you could ask, "Which team do you think will surprise us this year?" This invites more thoughtful responses.

3. **Highlight Unique Aspects:** Mention what makes this World Cup special. For instance, you could reference the multi-country hosting or the new teams that might be participating.

4. **Use a Hashtag Strategy:** In addition to the general hashtags, consider adding a couple of trending or more sp

In [10]:
for chunk in generate_chain.stream(
    {'messages': [request, AIMessage(content=tweet), HumanMessage(content=reflection)]}
):
    print(chunk.content, end='')

🌍⚽️ The countdown is ON! The FIFA World Cup 2026 is almost here, and it promises to be the most thrilling tournament yet across 🇺🇸🇨🇦🇲🇽! I'm rooting for [your favorite team]! 🏆 Which team do you think will surprise us this year? Share your World Cup memories and let's celebrate the beautiful game together! 🎉⚽️ #FIFAWorldCup2026 #WorldCup #FootballFever #SoccerCulture #MyWorldCupMemory 

Let the excitement begin! 🥳

## Define the Graph

In [16]:
# pip show langgraph

In [17]:
from typing import List, Sequence
from langgraph.graph import END, MessageGraph

In [ ]:
# defining a function for the generation node
def generation_node(state: Sequence[BaseMessage]):
    return generate_chain.invoke({'messages': state})

# defining a function for the reflection node
def reflection_node(messages: Sequence[BaseMessage]) -> List[BaseMessage]:
    # messages we need to adjust
    cls_map = {'ai': HumanMessage, 'human': AIMessage}
    # First message is the original user request. We keep it the same for all nodes
    translated = [messages[0]] + [
    cls_map[msg.type](content=msg.content) for msg in messages[1:]
    ]
    res = reflect_chain.invoke({'messages': translated})
    # We treat the output (AI message) of this as human feedback for the generator
    return HumanMessage(content=res.content)

# initializing the MessageGraph and adding two nodes to the graph: generate and reflect.
builder = MessageGraph()
builder.add_node('generate', generation_node)
builder.add_node('reflect', reflection_node)

# setting the generate node as the starting point
builder.set_entry_point('generate')

MAX_ITERATIONS = 5
def should_continue(state: List[BaseMessage]):
    if len(state) > MAX_ITERATIONS:
        return END
    return 'reflect'

# adding a conditional edge to the graph
builder.add_conditional_edges('generate', should_continue)
builder.add_edge('reflect', 'generate')

# compiling the graph
graph = builder.compile()

In [19]:
# !!! See below how to upgrade the code for the latest version of LangGraph and remove the warning above

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph().draw_mermaid_png()))

## Running the App

In [20]:
inputs = HumanMessage(content='Generate a tweed about FIFA World Cup 26')
response = graph.invoke(inputs)

In [22]:
for resp in response:
    print(resp.content)
    print('\n' + '-' * 100 + '\n')

Generate a tweed about FIFA World Cup 26

----------------------------------------------------------------------------------------------------

🌍⚽️ The countdown to #FIFAWorldCup26 has begun! With 48 teams and matches across the USA, Canada, and Mexico, this tournament is set to be the biggest yet! 🇺🇸🇨🇦🇲🇽 Which nation are you rooting for? Let the football fever take over! 🏆🔥 #WorldCup2026 #FootballFever

----------------------------------------------------------------------------------------------------

Your tweet has a solid foundation and captures the excitement around the upcoming FIFA World Cup 2026! Here’s a breakdown of what works and some suggestions to enhance its depth, style, and overall impact:

### Strengths:
1. **Emojis**: The use of emojis adds a visual element that captures attention and conveys emotion effectively.
2. **Hashtags**: You’ve incorporated relevant hashtags that will help in reaching a broader audience interested in the World Cup.
3. **Engagement**: Asking 

### Code Upgrade for the latest version of LangGraph

In [23]:
# 1. Define your state schema
# First, define a TypedDict to structure your state, with the "messages" key to accumulate messages via a reducer like add_messages.
# This ensures that each node’s messages get appended, not overwritten. 

from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

# Assume you already have these chains defined above:
# generate_chain, reflect_chain

# ---- State schema ----
class State(TypedDict):
    # Accumulate messages across nodes
    messages: Annotated[list[BaseMessage], add_messages]

    
# 2. Define your nodes as State → State transitions

# ---- Nodes ----
def generation_node(state: State) -> dict:
    """Call the generator with the running message history and append its reply."""
    ai_msg: BaseMessage = generate_chain.invoke({"messages": state["messages"]})
    return {"messages": [ai_msg]}

def reflection_node(state: State) -> dict:
    """
    Reflect on the conversation so far and append feedback as a HumanMessage
    (so the generator treats it like human guidance).
    """
    cls_map = {"ai": HumanMessage, "human": AIMessage}
    msgs = state["messages"]
    translated = [msgs[0]] + [cls_map[m.type](content=m.content) for m in msgs[1:]]
    res: BaseMessage = reflect_chain.invoke({"messages": translated})
    return {"messages": [HumanMessage(content=res.content)]}

# ---- Control flow ----
MAX_ITERATIONS = 3

def should_continue(state: State):
    """Stop after MAX_ITERATIONS generated messages, otherwise go to 'reflect'."""
    if len(state["messages"]) > MAX_ITERATIONS:
        return END
    return "reflect"


# 3. Build your StateGraph instead of MessageGraph

# ---- Build graph ----
builder = StateGraph(state_schema=State)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)

builder.set_entry_point("generate")
builder.add_conditional_edges("generate", should_continue)
builder.add_edge("reflect", "generate")

# ---- Compile ----
graph = builder.compile()


In [24]:
from langchain_core.messages import HumanMessage

# NEW: pass a state dict with a "messages" list
inputs = {"messages": [HumanMessage(content="Generate a tweet about FIFA World Cup 26")]}

# Invoke the graph and get the final state back
final_state = graph.invoke(inputs)

# Print all messages in the final state
for msg in final_state["messages"]:
    print(msg.content)
    print("\n" + "-" * 100 + "\n")


Generate a tweet about FIFA World Cup 26

----------------------------------------------------------------------------------------------------

🌍⚽️ Excitement is building for #FIFAWorldCup26! With matches across North America, it's not just a tournament—it's a celebration of culture, passion, and unity! 🇺🇸🇨🇦🇲🇽 Who are you cheering for? Let the countdown to glory begin! 🏆✨ #WorldCup2026 #FootballFever

----------------------------------------------------------------------------------------------------

Your tweet has a great foundation and captures the excitement surrounding the FIFA World Cup 2026. Here are some suggestions to enhance its depth, style, and overall impact:

1. **Add a Personal Touch**: Sharing a personal connection or memory related to the World Cup can resonate more with your audience. This could be a past experience or what the tournament means to you.

2. **Engage More Deeply**: Instead of just asking who people are cheering for, consider inviting them to share their